# Pipeline Completo: Análise Industrial de Países Emergentes
## Passos 1 a 9 - África e Médio Oriente

Este notebook executa todo o fluxo de dados com **geração automática de metadados** em cada passo:
- **Passo 1**: Extração de dados reais via API (WDI + WGI) + Metadados
- **Passo 2**: Análise Exploratória dos dados brutos
- **Passo 2.1**: Limpeza, Agregação (3 Métodos) e EDA Agregados + Metadados
- **Passo 3**: Engenharia de Features (A1, A2, A3) + Metadados
- **Passo 4**: Treinamento de 7 Modelos (5 Clássicos + 2 Bayesianos) + Metadados
- **Passo 5**: Avaliação de Performance
- **Passo 6**: Análise de Estratégias
- **Passo 7**: Interpretabilidade (SHAP)
- **Passo 8**: Análise Geográfica
- **Passo 9**: Análises Avançadas

### Modelos Treinados (7 total):
| # | Modelo | Tipo | Abordagem |
|---|--------|------|----------|
| 1 | Random Forest | Ensemble | Painel + Por País |
| 2 | XGBoost | Gradient Boosting | Painel + Por País |
| 3 | TFT (GradientBoosting) | Gradient Boosting | Painel + Por País |
| 4 | SARIMAX | Série Temporal (auto_arima) | Por País |
| 5 | LSTM (Keras/TensorFlow) | Rede Neural Recorrente | Por País (Janela Temporal) |
| 6 | Bayes Partial Pooling | Bayesiano Hierárquico | Hierárquico (PyMC) |
| 7 | Bayes Complete Pooling | Bayesiano Global | Global (PyMC) |

### Auto-Save para Google Drive:
Os resultados são salvos automaticamente no Google Drive após cada passo para evitar perda de dados.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 0: MONTAR GOOGLE DRIVE (AUTO-SAVE)
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob, json

# Pasta no Drive para salvar resultados
DRIVE_SAVE_DIR = '/content/drive/MyDrive/analise_industrial_resultados'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

def auto_save_to_drive(passo_nome, dirs_to_save):
    """Salva automaticamente os resultados de um passo no Google Drive."""
    passo_dir = os.path.join(DRIVE_SAVE_DIR, passo_nome)
    os.makedirs(passo_dir, exist_ok=True)
    n_files = 0
    for d in dirs_to_save:
        if os.path.exists(d):
            dest = os.path.join(passo_dir, os.path.basename(d))
            if os.path.isdir(d):
                shutil.copytree(d, dest, dirs_exist_ok=True)
            else:
                shutil.copy2(d, dest)
            n_files += 1
    # Salvar metadados JSON se existirem
    for meta in glob.glob(os.path.join(REPO_PATH, '**', 'metadata_*.json'), recursive=True):
        dest = os.path.join(passo_dir, os.path.basename(meta))
        shutil.copy2(meta, dest)
    print(f'  -> Auto-save: {n_files} itens salvos em {passo_dir}')

print(f'Google Drive montado com sucesso!')
print(f'Resultados serão salvos em: {DRIVE_SAVE_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 1: INSTALAÇÃO DE DEPENDÊNCIAS
# ═══════════════════════════════════════════════════════════════
!pip install wbgapi shap statsmodels pmdarima xgboost pymc arviz tensorflow geopandas -q
print('\nTodas as dependências instaladas com sucesso!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA 2: CLONAR REPOSITÓRIO E CONFIGURAR AMBIENTE
# ═══════════════════════════════════════════════════════════════
import os
import sys

# Clonar repositório (ignorar erro se já existir)
!git clone https://github.com/ottrindade1963/analise-industrialX.git 2>/dev/null || echo 'Repositório já existe'

# Detectar automaticamente o nome do diretório clonado
dirs = [d for d in os.listdir('/content') if os.path.isdir(f'/content/{d}') and d not in ['.config', 'sample_data', 'drive']]
repo_dir = dirs[0] if dirs else 'analise-industrialX'
REPO_PATH = f'/content/{repo_dir}'

# Mudar para o diretório do repositório
%cd {REPO_PATH}

# Adicionar ao sys.path para imports
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Verificar ficheiros principais
print(f'\nDiretório atual: {os.getcwd()}')
py_files = [f for f in os.listdir() if f.endswith('.py') and not f.startswith('test')]
print(f'Ficheiros Python: {len(py_files)}')
for key_file in ['metadata_generator.py', 'passo4_bayesian_model.py', 'train_single_dataset.py', 'passo4_model_train_pipeline.py']:
    status = 'OK' if key_file in py_files else 'FALTA'
    print(f'  {key_file}: {status}')

# Guardar caminho para uso posterior
%store REPO_PATH

---
## Passo 1: Extração de Dados Reais (WDI + WGI via API)
Extrai dados quantitativos (WDI) e qualitativos (WGI) da API do Banco Mundial.
Gera `metadata_passo1.json` com fontes, datas de download e cobertura por indicador.

In [ ]:
%cd {REPO_PATH}
!python3 passo1_master_pipeline.py

# Auto-save Passo 1
auto_save_to_drive('passo1_extracao', ['data/raw'])

In [ ]:
# Verificar metadados gerados no Passo 1
meta1_path = os.path.join(REPO_PATH, 'data', 'raw', 'metadata_passo1.json')
if os.path.exists(meta1_path):
    with open(meta1_path) as f:
        meta1 = json.load(f)
    print('METADADOS DO PASSO 1:')
    print(f'  Países extraídos: {meta1["paises"]["total_extraidos"]}')
    print(f'  Indicadores WDI: {meta1["indicadores_wdi"]["total"]}')
    print(f'  Indicadores WGI: {meta1["indicadores_wgi"]["total"]}')
    print(f'  WDI dimensões: {meta1["dimensoes"]["wdi"]}')
    print(f'  WGI dimensões: {meta1["dimensoes"]["wgi"]}')
else:
    print('AVISO: Metadados do Passo 1 não encontrados')
    print(f'  Procurado em: {meta1_path}')

---
## Passo 2: Análise Exploratória dos Dados Brutos

In [ ]:
%cd {REPO_PATH}
!python3 passo2_master_pipeline.py

# Auto-save Passo 2
auto_save_to_drive('passo2_eda', ['resultados_eda', 'resultados_eda_quali'])

In [ ]:
# Visualizações do Passo 2
from IPython.display import Image, display

print('=== EDA QUANTITATIVA ===')
eda_quant_dir = os.path.join(REPO_PATH, 'resultados_eda')
if os.path.exists(eda_quant_dir):
    for f in sorted(os.listdir(eda_quant_dir)):
        if f.endswith('.png'):
            print(f'\n{f}')
            display(Image(os.path.join(eda_quant_dir, f), width=800))

print('\n=== EDA QUALITATIVA ===')
eda_quali_dir = os.path.join(REPO_PATH, 'resultados_eda_quali')
if os.path.exists(eda_quali_dir):
    for f in sorted(os.listdir(eda_quali_dir)):
        if f.endswith('.png'):
            print(f'\n{f}')
            display(Image(os.path.join(eda_quali_dir, f), width=800))

---
## Passo 2.1: Limpeza, Agregação (3 Métodos) e EDA Agregados
Limpa dados, aplica imputação por variável, agrega WDI+WGI por 3 métodos (Inner, Left, Outer).
Gera `metadata_passo2_1.json` com missing antes/depois, métodos de imputação e dimensões dos agregados.

In [ ]:
%cd {REPO_PATH}
!python3 passo2_1_master_pipeline.py

# Auto-save Passo 2.1
auto_save_to_drive('passo2_1_limpeza', [
    'dados_limpos',
    'agregado_metodo1_inner', 'agregado_metodo2_left_imputado', 'agregado_metodo3_outer_completo',
    'eda_nao_agregados_limpos', 'eda_metodo1_inner', 'eda_metodo2_left', 'eda_metodo3_outer'
])

In [ ]:
# Verificar metadados gerados no Passo 2.1
meta21_path = os.path.join(REPO_PATH, 'dados_limpos', 'metadata_passo2_1.json')
if os.path.exists(meta21_path):
    with open(meta21_path) as f:
        meta21 = json.load(f)
    print('METADADOS DO PASSO 2.1:')
    print(f'  WDI original: {meta21["limpeza_wdi"]["dimensoes_originais"]}')
    print(f'  WDI após limpeza: {meta21["limpeza_wdi"]["dimensoes_apos_limpeza"]}')
    print(f'  Linhas removidas WDI: {meta21["limpeza_wdi"]["linhas_removidas"]}')
    print(f'  Métodos de imputação: {meta21["limpeza_wdi"]["metodos_imputacao"]}')
    print(f'\n  Agregação:')
    for metodo, info in meta21['agregacao'].items():
        print(f'    {metodo}: {info["descricao"]}')
        if info.get('resultado'):
            print(f'      -> {info["resultado"]}')
else:
    print('AVISO: Metadados do Passo 2.1 não encontrados')
    print(f'  Procurado em: {meta21_path}')

---
## Passo 3: Engenharia de Features
Aplica 3 estratégias de engenharia de features (A1: Direta, A2: PCA, A3: Interação) aos 4 datasets.
Gera `metadata_passo3.json` com features criadas, PCA variância explicada e top correlações.

In [ ]:
%cd {REPO_PATH}
!python3 passo3_feat_eng_pipeline.py

# Auto-save Passo 3
auto_save_to_drive('passo3_engenharia', ['dados_engenharia'])

In [ ]:
# Verificar metadados gerados no Passo 3
meta3_path = os.path.join(REPO_PATH, 'dados_engenharia', 'metadata_passo3.json')
if os.path.exists(meta3_path):
    with open(meta3_path) as f:
        meta3 = json.load(f)
    print('METADADOS DO PASSO 3:')
    print(f'  Estratégias: {list(meta3["estrategias"].keys())}')
    print(f'\n  Datasets gerados:')
    for ds, strats in meta3['datasets_gerados'].items():
        for st, info in strats.items():
            print(f'    {ds}/{st}: shape={info["shape"]}, features={info["n_features"]}')
    if meta3.get('pca_info'):
        print(f'\n  PCA Info: {meta3["pca_info"]}')
else:
    print('AVISO: Metadados do Passo 3 não encontrados')
    print(f'  Procurado em: {meta3_path}')

---
## Passo 4: Treinamento de 7 Modelos (5 Clássicos + 2 Bayesianos)

Treina 7 modelos em 10 combinações dataset x estratégia (70 combinações total):
- **Clássicos**: RandomForest, XGBoost, TFT, SARIMAX (auto_arima), LSTM (Keras real)
- **Bayesianos**: Partial Pooling (Hierárquico), Complete Pooling (Global)

Divisão temporal: Treino <= 2016 | Validação 2017-2019 | Teste >= 2020

**Subprocess Isolation**: Cada dataset é treinado num subprocesso Python separado.
Quando o subprocesso termina, TODA a memória é libertada. Se crashar, basta executar novamente.

**Tempo estimado**: ~40-60 min por dataset x 10 datasets = ~6-8 horas

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PASSO 4: VERIFICAR PROGRESSO E ESCOLHER OPÇÃO
# ═══════════════════════════════════════════════════════════════
%cd {REPO_PATH}

modelos_dir = os.path.join(REPO_PATH, 'modelos_treinados')
summaries_existentes = glob.glob(os.path.join(modelos_dir, '*_summary.json')) if os.path.exists(modelos_dir) else []
modelos_pkl = glob.glob(os.path.join(modelos_dir, '*.pkl')) if os.path.exists(modelos_dir) else []

if summaries_existentes:
    print(f'{len(summaries_existentes)} datasets já treinados (serão saltados automaticamente):')
    for sf in sorted(summaries_existentes):
        print(f'    - {os.path.basename(sf).replace("_summary.json", "")}')
    print(f'\n  PKL existentes: {len(modelos_pkl)}')
    print(f'\nEscolha uma opção:')
    print('1 - Continuar treinamento (retomar onde parou)')
    print('2 - Treinar tudo do zero (apaga resultados anteriores)')
    print('3 - Apenas gerar visualizações com modelos existentes')
    opcao = input('Digite 1, 2 ou 3: ').strip()
    if opcao == '2':
        shutil.rmtree(modelos_dir, ignore_errors=True)
        print('\nResultados anteriores apagados. Treinamento do zero.')
    TREINAR_MODELOS = opcao in ('1', '2')
else:
    print('Nenhum modelo encontrado. Será necessário treinar.')
    TREINAR_MODELOS = True

if TREINAR_MODELOS:
    print(f'\nTempo estimado: ~40-60 min por dataset x 10 datasets = ~6-8 horas')
    print(f'Se crashar, basta executar esta célula novamente (retoma automaticamente)')
    print(f'Resultados são salvos no Google Drive após cada dataset')
else:
    print(f'\nOpção: Apenas visualizações')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXECUTAR PASSO 4 (SUBPROCESS ISOLATION + AUTO-SAVE)
# ═══════════════════════════════════════════════════════════════
%cd {REPO_PATH}

if TREINAR_MODELOS:
    print('\n' + '='*70)
    print('TREINAMENTO COM SUBPROCESS ISOLATION')
    print('  Cada dataset corre num subprocesso isolado (sem memory leak)')
    print('  RF(100iter), XGBoost(100iter+EarlyStopping), TFT(100iter)')
    print('  SARIMAX(auto_arima), LSTM(Keras real), Bayesianos(MCMC)')
    print('  Se crashar, execute novamente - retoma automaticamente')
    print('='*70)
    !python3 passo4_model_train_pipeline.py
else:
    print('\n' + '='*70)
    print('GERANDO APENAS VISUALIZAÇÕES')
    print('='*70)
    from passo4_model_train_visualizer import TrainingVisualizer
    viz = TrainingVisualizer()
    print('\n[1/4] Gerando gráficos de métricas de treinamento...')
    viz.plot_real_training_metrics()
    print('[2/4] Gerando gráficos de previsões vs valores reais...')
    viz.plot_predictions_vs_actual()
    print('[3/4] Gerando análise detalhada do melhor modelo...')
    viz.plot_best_model_analysis()
    print('[4/4] Gerando comparação de previsões...')
    viz.plot_predictions_comparison()
    print('\nVisualizações geradas com sucesso!')

# Auto-save Passo 4
auto_save_to_drive('passo4_treinamento', ['modelos_treinados'])

In [ ]:
# Verificar metadados gerados no Passo 4
meta4_path = os.path.join(REPO_PATH, 'modelos_treinados', 'metadata_passo4.json')
if os.path.exists(meta4_path):
    with open(meta4_path) as f:
        meta4 = json.load(f)
    print('METADADOS DO PASSO 4:')
    print(f'  Modelos: {list(meta4["modelos"].keys())}')
    print(f'  Tempo total: {meta4["tempos_treino"]}')
    print(f'  Divisão temporal: {meta4["configuracao"]["divisao_temporal"]}')
    if 'melhor_modelo_global' in meta4:
        best = meta4['melhor_modelo_global']
        print(f'\n  MELHOR MODELO GLOBAL:')
        print(f'     Modelo: {best["modelo"]}')
        print(f'     Dataset: {best["dataset"]}')
        print(f'     Estratégia: {best["estrategia"]}')
        print(f'     R2: {best["r2"]:.4f}')
        print(f'     RMSE: {best["rmse"]:.4f}')
    if 'ranking_modelos' in meta4:
        print(f'\n  RANKING DE MODELOS (por R2 médio):')
        ranking = sorted(meta4['ranking_modelos'].items(), key=lambda x: x[1].get('mean', -999), reverse=True)
        for i, (modelo, stats) in enumerate(ranking, 1):
            print(f'     {i}. {modelo}: média={stats.get("mean", "N/A")}, max={stats.get("max", "N/A")}')
else:
    print('AVISO: Metadados do Passo 4 não encontrados')
    print(f'  Procurado em: {meta4_path}')

In [ ]:
# Tabela comparativa de resultados do Passo 4
import pandas as pd

summary_path = os.path.join(REPO_PATH, 'modelos_treinados', 'resumo_treinamento_completo.csv')
if os.path.exists(summary_path):
    df_summary = pd.read_csv(summary_path)
    print('\nTABELA COMPARATIVA COMPLETA (Top 20 por R2 Global):')
    top20 = df_summary.sort_values('Global_R2', ascending=False).head(20)
    display(top20[['Dataset', 'Estrategia', 'Modelo', 'Global_R2', 'Global_RMSE', 'Global_MAE', 'PerCountry_Median_R2', 'N_Countries']].reset_index(drop=True))
    
    print('\nMELHOR MODELO POR TIPO:')
    best_per_model = df_summary.loc[df_summary.groupby('Modelo')['Global_R2'].idxmax()]
    display(best_per_model[['Modelo', 'Dataset', 'Estrategia', 'Global_R2', 'Global_RMSE', 'PerCountry_Median_R2']].sort_values('Global_R2', ascending=False).reset_index(drop=True))
else:
    alt_path = os.path.join(REPO_PATH, 'modelos_treinados', 'tabela_comparativa_completa.csv')
    if os.path.exists(alt_path):
        df_alt = pd.read_csv(alt_path)
        display(df_alt.sort_values('Global_R2', ascending=False).head(20))
    else:
        print('AVISO: Tabela comparativa não encontrada. Execute o treinamento primeiro.')

In [ ]:
# Visualizações do Passo 4
viz_dir = os.path.join(REPO_PATH, 'modelos_treinados', 'visualizacoes_treino')
if os.path.exists(viz_dir):
    files = sorted([f for f in os.listdir(viz_dir) if f.endswith('.png')])
    print(f'Total de visualizações geradas: {len(files)}')
    for f in files:
        print(f'\n{f}')
        display(Image(os.path.join(viz_dir, f), width=800))
else:
    print('AVISO: Diretório de visualizações não encontrado')

---
## Passo 5: Avaliação de Performance

In [ ]:
%cd {REPO_PATH}
!python3 passo5_eval_pipeline.py

# Auto-save Passo 5
auto_save_to_drive('passo5_avaliacao', ['resultados_avaliacao'])

In [ ]:
# Visualizações do Passo 5
viz_dir = os.path.join(REPO_PATH, 'resultados_avaliacao', 'visualizacoes_avaliacao')
if os.path.exists(viz_dir):
    for f in sorted(os.listdir(viz_dir)):
        if f.endswith('.png'):
            print(f'\n{f}')
            display(Image(os.path.join(viz_dir, f), width=800))

---
## Passo 6: Análise de Estratégias

In [ ]:
%cd {REPO_PATH}
!python3 passo6_strategy_pipeline.py

# Auto-save Passo 6
auto_save_to_drive('passo6_estrategias', ['analise_estrategias'])

In [ ]:
# Visualizações do Passo 6
viz_dir = os.path.join(REPO_PATH, 'analise_estrategias', 'visualizacoes_estrategias')
if os.path.exists(viz_dir):
    for f in sorted(os.listdir(viz_dir)):
        if f.endswith('.png'):
            print(f'\n{f}')
            display(Image(os.path.join(viz_dir, f), width=800))

---
## Passo 7: Interpretabilidade (SHAP)

In [ ]:
%cd {REPO_PATH}
!python3 passo7_shap_pipeline.py

# Auto-save Passo 7
auto_save_to_drive('passo7_shap', ['interpretabilidade_shap'])

In [ ]:
# Visualizações do Passo 7 (Primeiras 10)
viz_dir = os.path.join(REPO_PATH, 'interpretabilidade_shap', 'visualizacoes_shap')
if os.path.exists(viz_dir):
    files = sorted([f for f in os.listdir(viz_dir) if f.endswith('.png')])
    for f in files[:10]:
        print(f'\n{f}')
        display(Image(os.path.join(viz_dir, f), width=800))
    if len(files) > 10:
        print(f'\n... e mais {len(files)-10} gráficos SHAP gerados.')

---
## Passo 8: Análise Geográfica

In [ ]:
%cd {REPO_PATH}
!python3 passo8_geo_pipeline.py

# Auto-save Passo 8
auto_save_to_drive('passo8_geografica', ['analise_geografica'])

In [ ]:
# Visualizações do Passo 8 (Primeiras 10)
viz_dir = os.path.join(REPO_PATH, 'analise_geografica', 'visualizacoes_geograficas')
if os.path.exists(viz_dir):
    files = sorted([f for f in os.listdir(viz_dir) if f.endswith('.png')])
    for f in files[:10]:
        print(f'\n{f}')
        display(Image(os.path.join(viz_dir, f), width=800))
    if len(files) > 10:
        print(f'\n... e mais {len(files)-10} gráficos geográficos gerados.')

---
## Passo 9: Análises Avançadas

In [ ]:
%cd {REPO_PATH}
!python3 passo9_advanced_pipeline.py

# Auto-save Passo 9
auto_save_to_drive('passo9_avancadas', ['analises_avancadas'])

In [ ]:
# Visualizações do Passo 9
viz_dir = os.path.join(REPO_PATH, 'analises_avancadas', 'visualizacoes_avancadas')
if os.path.exists(viz_dir):
    for f in sorted(os.listdir(viz_dir)):
        if f.endswith('.png'):
            print(f'\n{f}')
            display(Image(os.path.join(viz_dir, f), width=800))

---
## Resumo Final e Metadados Consolidados

In [ ]:
%cd {REPO_PATH}

print('=' * 70)
print('RESUMO FINAL DO PIPELINE')
print('=' * 70)

# Contagens
n_png = len(glob.glob('**/*.png', recursive=True))
n_csv = len(glob.glob('**/*.csv', recursive=True))
n_pkl = len(glob.glob('**/*.pkl', recursive=True))
n_json = len(glob.glob('**/metadata_*.json', recursive=True))

print(f'\n  Visualizações (PNG): {n_png}')
print(f'  Datasets (CSV): {n_csv}')
print(f'  Modelos (PKL): {n_pkl}')
print(f'  Metadados (JSON): {n_json}')

# Listar metadados
print(f'\n  FICHEIROS DE METADADOS GERADOS:')
for meta_file in sorted(glob.glob('**/metadata_*.json', recursive=True)):
    size = os.path.getsize(meta_file)
    print(f'    {meta_file} ({size:,} bytes)')

# Listar modelos PKL
print(f'\n  MODELOS TREINADOS:')
for pkl_file in sorted(glob.glob('modelos_treinados/*.pkl')):
    size = os.path.getsize(pkl_file)
    name = os.path.basename(pkl_file)
    print(f'    {name} ({size/1024:.0f} KB)')

print(f'\n{"=" * 70}')
print('PIPELINE COMPLETO EXECUTADO COM SUCESSO!')
print(f'{"=" * 70}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SAVE FINAL: Salvar TUDO no Google Drive + Download ZIP
# ═══════════════════════════════════════════════════════════════
%cd {REPO_PATH}

# 1. Salvar tudo no Drive (cópia completa)
final_dir = os.path.join(DRIVE_SAVE_DIR, 'COMPLETO')
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)

dirs_to_save = [
    'data/raw', 'dados_limpos', 'dados_engenharia',
    'agregado_metodo1_inner', 'agregado_metodo2_left_imputado', 'agregado_metodo3_outer_completo',
    'modelos_treinados', 'resultados_eda', 'resultados_eda_quali',
    'eda_nao_agregados_limpos', 'eda_metodo1_inner', 'eda_metodo2_left', 'eda_metodo3_outer',
    'resultados_avaliacao', 'analise_estrategias', 'interpretabilidade_shap',
    'analise_geografica', 'analises_avancadas'
]

os.makedirs(final_dir, exist_ok=True)
for d in dirs_to_save:
    if os.path.exists(d):
        dest = os.path.join(final_dir, d)
        shutil.copytree(d, dest, dirs_exist_ok=True)

# Copiar metadados e scripts Python
for f in glob.glob('*.py') + glob.glob('**/metadata_*.json', recursive=True):
    dest = os.path.join(final_dir, f)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(f, dest)

print(f'Todos os resultados salvos no Google Drive: {final_dir}')

# 2. Criar ZIP para download local
output_zip = '/content/resultados_pipeline_completo'
shutil.make_archive(output_zip, 'zip', final_dir)
zip_size = os.path.getsize(output_zip + '.zip') / 1024 / 1024
print(f'\nZIP criado: {output_zip}.zip ({zip_size:.1f} MB)')

# 3. Copiar ZIP para o Drive também
shutil.copy2(output_zip + '.zip', os.path.join(DRIVE_SAVE_DIR, 'resultados_pipeline_completo.zip'))
print(f'ZIP copiado para o Drive: {DRIVE_SAVE_DIR}/resultados_pipeline_completo.zip')

# 4. Auto-download no Colab
try:
    from google.colab import files
    files.download(output_zip + '.zip')
except:
    print('\nPara download manual: navegue até /content/ no painel de ficheiros à esquerda')